# Joining datasets
we are using this to merge the 2011-2020 and 2020-2026 data. 

In [1]:
#importing libraries
import pandas as pd 
import numpy as np

In [2]:
#importing files
df_complaints_2011_2020 = pd.read_csv('../Joining Datasets/Data/iad_extracted_with_youth_labels_scrubbed.csv')

df_complaints_2020_2026 = pd.read_excel('../Joining Datasets/Data/Mikaela McSharry MuckRock B2026-519.xlsx')

In [3]:
df_complaints_2020_2026.head()

,Inc: IA No,Inc: Incident type,Inc: Received date,Off: Title/rank,Off: First name,Off: Last name,Off: Employee ID Number,Alg: Allegation,Alg: Finding,Act: Action taken,Act: Days/hours suspended,Inc: Narrative
0,IAD2020-0379,Citizen complaint,09/11/2020,Police Officer,Michael J,Geary,135954,Neg.Duty/Unreasonable Judge,Sustained,Administrative Hearing Decision - Termination,0,An anonymous caller contacted IAD and stated ...
1,IAD2021-0009,Citizen complaint,01/07/2021,Police Officer,Michael J,Geary,135954,Conduct Unbecoming,Sustained,Administrative Hearing Decision - Termination,0,The complainant who would like to stay private...
2,IAD2022-0302,Citizen complaint,07/25/2022,Police Officer\nPolice Officer\nPolice Officer...,Kevin K\nKevin K\nKevin K\nKevin K,Herbert\nHerbert\nHerbert\nHerbert,148269\n148269\n148269\n148269,Self Identification\nSelf Identification\nNeg....,Sustained\nSustained\nSustained\nSustained,Administrative Hearing Decision\nSuspension\nA...,0\n3\n0\n3,"Last night(Friday 7/22/22), I was followed and..."
3,IAD2022-0399,Citizen complaint,10/03/2022,Police Officer\nPolice Officer\nPolice Officer,James C\nJames C\nMatthew James,Hosea\nHosea\nBeggan,128227\n128227\n112120,Body Worn Policy - Activation & Incidents of U...,Sustained\nSustained\nSustained,Administrative Hearing Decision\nSuspension\nS...,0\n1\n1,I the writer JasonWright would like to file a ...
4,IAD2022-0378,Internal investigation,09/27/2022,Police Officer\nPolice Officer\nPolice Officer...,Hena V\nHena V\nHena V\nHena V,Lopes\nLopes\nLopes\nLopes,148288\n148288\n148288\n148288,Body Worn Camera Deployment - Officer Responsi...,Sustained\nSustained\nSustained\nSustained,Administrative Hearing\nSuspension\nAdministra...,0\n2\n0\n2,During an active IAD investigation it was dete...


In [4]:
# Columns where one cell holds several newline-separated values (aligned by position)
multi_cols = [
    "Off: Title/rank", "Off: First name", "Off: Last name", "Off: Employee ID Number",
    "Alg: Allegation", "Alg: Finding", "Act: Action taken", "Act: Days/hours suspended",
]
 
def split_cell(v):
    if pd.isna(v):
        return [None]
    return [p.strip() for p in str(v).replace("\r", "").split("\n")]
 
for c in multi_cols:
    df_complaints_2020_2026[c] = df_complaints_2020_2026[c].apply(split_cell)
 
# Check that every multi-value column has the same item count within each row
lens = df_complaints_2020_2026[multi_cols].apply(lambda col: col.str.len())
bad = lens[lens.nunique(axis=1) > 1]
if len(bad):
    print("WARNING: mismatched item counts in these rows (check by hand):")
    print(df_complaints_2020_2026.loc[bad.index, "Inc: IA No"].tolist())
    # pad shorter lists with None so explode doesn't fail
    n = lens.max(axis=1)
    for c in multi_cols:
        df_complaints_2020_2026[c] = [l + [None] * (m - len(l)) for l, m in zip(df_complaints_2020_2026[c], n)]
 


In [5]:
out = df_complaints_2020_2026.explode(multi_cols, ignore_index=True)
out["Inc: Narrative"] = (
    out["Inc: Narrative"]
    .str.replace(r"\s*[\r\n]+\s*", " ¶ ", regex=True)
)

In [6]:
# renaming columns to match the 2011-2020 dataset
out = out.rename(columns={
    "Inc: IA No": "ia_no",
    "Inc: Incident type": "incident_type_x",
    "Inc: Received date": "received_date_x",
    "Inc: Narrative": "narrative_x",
    "Alg: Allegation": "allegation_x",
    "Alg: Finding": "finding_x"
})

In [7]:
out.to_csv("Data/2020_2026_exploded.csv", index=False)